In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

In [6]:
import re
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from src.schemas import dispatch
from src.schemas.banks.pagcorp import Pagcorp
from src.schemas.parsers.pdf_extractor import PDFExtractor

# origem = r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\00_ABASE\Lançamentos_Contabeis.xls"
# name = "[LANC] - Extrato_890000983290_03-06-2026_Parte1 (1).xls".replace("EXT", "LANC")
# destino = rf"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\EXTRATOS 0626\{name}.xls"

# shutil.copy2(origem, destino)

ext = r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\0426_EXTBAN NUBANK_PURE.xlsx"

path = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\0426_EXTBAN NUBANK_PURE.pdf")

if path.suffix == ".xlsx":
    df = pd.read_excel(path)
    df = Pagcorp(df).layout1()
else:
    pdf = PDFExtractor(path).extract()

if pdf:
    #df = dispatch(pdf)
    display(pdf)


['PURE OFIR CORRETORA DE SEGUROS LTDA',
 'CNPJ 62.206.728/0001-21 Agência 0001 Conta',
 '388191222-7',
 '01 DE ABRIL DE 2026 a 30 DE ABRIL DE 2026 VALORES EM R$',
 'Saldo inicial 14,76',
 'Rendimento líquido +0,00',
 'Saldo final do período',
 'R$ 14,76 Total de entradas +0,00',
 'Total de saídas -0,00',
 'Saldo final do período 14,76',
 'Nenhuma movimentação realizada.',
 'O saldo líquido corresponde ao total de depósitos e rendimentos em conta, não considerando movimentações feitas após a data mencionada.',
 'Não nos responsabilizamos pelo uso indevido ou por alterações das informações originalmente contidas neste documento após envio.',
 'Asseguramos a autenticidade destas movimentações e das informações aqui citadas.',
 'Nu Financeira S.A. - Sociedade de Credito, Financiamento',
 'Nu Pagamentos S.A. - Instituição de Pagamento',
 'e Investimento',
 'CNPJ: 18.236.120/0001-58',
 'CNPJ: 30.680.829/0001-43',
 'Tem alguma dúvida? Mande uma mensagem para nosso time de atendimento pelo cha

In [ ]:
empresa = pdf[0]
pdf = pdf[11:]
pdf = [item for item in pdf if not any(texto in item for texto in [empresa, "CNPJ", "Tem alguma dúvida?", "metropolitanas) ", "Caso a solução ", "disponíveis em nubank.com.br", "Extrato gerado ", "O saldo líquido ", "Não nos responsabilizamos ", "Asseguramos a autenticidade ", "Nu Financeira S.A.", "e Investimento", "Nu Pagamentos S.A. - Instituição de Pagamento"])]

rx_data = re.compile(
    r"^(?P<data>(?:\d{2}/\d{2}(?:/\d{2,4})?|\d{2}\s+"
    r"(?:JAN|FEV|MAR|ABR|MAI|JUN|JUL|AGO|SET|OUT|NOV|DEZ)\s+\d{4}))\b",
    re.I
)

rx_valor = re.compile(
    r"(?P<valor>(?:R\$\s*)?[+-]?\s*(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}[CD]?)$",
    re.I
)

rx_inicio_mov = re.compile(
    r"^(?:Transferência enviada pelo Pix|Transferência Recebida|Transferência recebida pelo pix|"
    r"Crédito em conta|Pagamento de boleto efetuado)\b",
    re.I
)

rx_total_entrada = re.compile(r"^Total de entradas\b", re.I)
rx_total_saida = re.compile(r"^Total de saídas\b", re.I)
rx_separador = re.compile(r"^(?:Saldo do dia|Saldo final|Saldo inicial)\b", re.I)

rx_lixo_linha = re.compile(
    r"^(?:Movimentações|CNPJ\b|Agência\b|Conta\b|VALORES EM R\$|"
    r"Tem alguma dúvida|Extrato gerado|O saldo líquido|"
    r"Não nos responsabilizamos|Asseguramos|Nu Financeira|"
    r"Nu Pagamentos S\.A)",
    re.I
)

rx_lixo_meio = re.compile(
    r"\b(?:ADVOCACIA\s+364970674-3\s+)?"
    r"(?:\d{2}\s+DE\s+[A-ZÇ]+\s+DE\s+\d{4}\s+a\s+"
    r"\d{2}\s+DE\s+[A-ZÇ]+\s+DE\s+\d{4}\s+VALORES\s+EM\s+R\$)"
    r"|\bADVOCACIA\s+364970674-3\b",
    re.I
)

def limpar(texto: str) -> str:
    texto = rx_lixo_meio.sub(" ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def tipo_pela_descricao(descricao: str, tipo_bloco: str) -> str:
    if re.search(r"^(?:Transferência Recebida|Transferência recebida pelo pix|Crédito em conta)\b", descricao, re.I):
        return "entrada"

    if re.search(r"^(?:Transferência enviada pelo Pix|Transferência recebida pelo pix|Pagamento de boleto efetuado)\b", descricao, re.I):
        return "saida"

    return tipo_bloco

def ajustar_sinal(valor: str, descricao: str, tipo_bloco: str) -> str:
    tipo = tipo_pela_descricao(descricao, tipo_bloco)
    valor_sem_sinal = re.sub(r"^[+-]\s*", "", valor.strip())

    if tipo == "saida":
        return "-" + valor_sem_sinal

    if tipo == "entrada":
        return valor_sem_sinal

    return valor.strip()

def montar(data: str, partes: list[str], valor: str, tipo_bloco: str) -> str:
    descricao = limpar(" ".join(partes))
    valor = ajustar_sinal(valor, descricao, tipo_bloco)
    return limpar(f"{data} {descricao} {valor}")

resultado = []
data_atual = ""
partes = []
valor = ""
tipo_bloco = ""

for item in pdf:
    linha = limpar(item)

    if not linha:
        continue

    m_data = rx_data.match(linha)
    if m_data:
        if data_atual and partes and valor:
            resultado.append(montar(data_atual, partes, valor, tipo_bloco))

        data_atual = m_data.group("data")
        partes = []
        valor = ""
        linha = limpar(linha[m_data.end():])

        if not linha:
            continue

    if rx_total_entrada.match(linha):
        if data_atual and partes and valor:
            resultado.append(montar(data_atual, partes, valor, tipo_bloco))
        tipo_bloco = "entrada"
        partes = []
        valor = ""
        continue

    if rx_total_saida.match(linha):
        if data_atual and partes and valor:
            resultado.append(montar(data_atual, partes, valor, tipo_bloco))
        tipo_bloco = "saida"
        partes = []
        valor = ""
        continue

    if rx_separador.match(linha):
        if data_atual and partes and valor:
            resultado.append(montar(data_atual, partes, valor, tipo_bloco))
        partes = []
        valor = ""
        tipo_bloco = ""
        continue

    if rx_lixo_linha.search(linha):
        continue

    if rx_inicio_mov.match(linha):
        if data_atual and partes and valor:
            resultado.append(montar(data_atual, partes, valor, tipo_bloco))
        partes = []
        valor = ""

    if not partes and not rx_inicio_mov.match(linha):
        continue

    m_valor = rx_valor.search(linha)

    if m_valor:
        valor = m_valor.group("valor").strip()
        texto_sem_valor = limpar(linha[:m_valor.start()])
        if texto_sem_valor:
            partes.append(texto_sem_valor)
    else:
        partes.append(linha)

if data_atual and partes and valor:
    resultado.append(montar(data_atual, partes, valor, tipo_bloco))

padrao = r"^(\d{2}\s+[A-Z]{3}\s+\d{4})\s+(.+?)\s+(-?\s?\d{1,3}(?:\.\d{3})*,\d{2})$"

dados = []

for item in resultado:
    match = re.match(padrao, item)

    if match:
        data = match.group(1)
        descricao = match.group(2).strip()
        valor = match.group(3).strip()

        dados.append({
            "DATA": data,
            "DESCRIÇÃO": descricao,
            "VALOR": valor
        })
    else:
        dados.append({
            "DATA": None,
            "DESCRIÇÃO": item,
            "VALOR": None
        })
df = pd.json_normalize(dados)
if not df.empty:
    continue
display(dados)

[]

In [ ]:

import shutil
import pandas as pd
from src.schemas import dispatch
from src.schemas.banks.pagcorp import Pagcorp
from src.schemas.parsers.pdf_extractor import PDFExtractor
from src.utils.helpers import totalizador, planilha_lancamento

fecfin = r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\0426_FECFIN_CEMAF OPE.xlsx"
df_sicoob = pd.read_excel(fecfin, sheet_name="SICOOB 18723-2")
df_sicoob.columns = df_sicoob.iloc[5]
df_sicoob = df_sicoob[6:].reset_index(drop=True)
df_sicoob = df_sicoob.dropna(subset=["DATA"])
df_sicoob = df_sicoob[~df_sicoob["HISTÓRICO"].astype(str).str.upper().str.contains("SALDO", na=False)]
df_sicoob["SAIDA"] = df_sicoob["SAIDA"] * -1
df_sicoob["DESCRIÇÃO"] = df_sicoob.apply(lambda row: f"{row["TIPO"]} {row["Nº DOC"]} {row["HISTÓRICO"]} {row["OBS P/ CONT"]} {row["OBS P/ INTERNAS"]}", axis=1)
df_sicoob["DESCRIÇÃO"] = (df_sicoob["DESCRIÇÃO"].str.strip().str.replace("nan", "").str.replace("  ", " "))
df_sicoob["VALOR"] = df_sicoob.apply(lambda row: row["ENTRADA"] if row["ENTRADA"] > 0 else row["SAIDA"], axis=1)
df_sicoob = df_sicoob[["DATA", "DESCRIÇÃO", "VALOR"]]
df_sicoob["TIPO"] = df_sicoob.apply(lambda x: "C" if x["VALOR"] > 0 else "D", axis=1)

planilha_lancamento(df_sicoob, )
  
display(df_sicoob)


In [ ]:
from src.utils.helpers import totalizador, planilha_lancamento

planilha_lancamento(df, destino)
df = totalizador(df)
df.to_excel(ext, index=False)

display(df)

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
from src.schemas import dispatch
from src.schemas.parsers.pdf_extractor import PDFExtractor
from src.utils.helpers import totalizador, planilha_lancamento


lancamento = Path.cwd() / r"data\Lancamentos_Contabeis.xls"
extratos = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT")

invalidos = Path(extratos / "00_INVALIDOS")
convertidos = Path(extratos / "00_CONVERTIDOS")
invalidos.mkdir(parents=True, exist_ok=True)
convertidos.mkdir(parents=True, exist_ok=True)

if extratos.exists():
    arquivos = [arquivo for arquivo in extratos.iterdir()]
    for arquivo in arquivos:
        if arquivo.is_file() and arquivo.suffix.lower() == ".pdf" and "EXT" in arquivo.stem:
            
            pdf = PDFExtractor(arquivo).extract()

            if not pdf:
                destino = invalidos / arquivo.name
                shutil.move(str(arquivo), str(destino))
                print(f"PDF movido para inválidos porque está vazio: {arquivo.name}")
                continue

            df = dispatch(pdf)

            if df is None or df.empty:
                destino = invalidos / arquivo.name
                shutil.move(str(arquivo), str(destino))
                print(f"PDF movido para inválidos porque o DataFrame veio vazio: {arquivo.name}")
                continue

            # Cria uma pasta dentro de 00_CONVERTIDOS com o nome do arquivo
            pasta_arquivo = convertidos / arquivo.stem
            pasta_arquivo.mkdir(parents=True, exist_ok=True)

            name = f"{arquivo.stem}".replace("EXT", "LANC")

            dest_lancamento = pasta_arquivo / f"{name}.xls"
            dest_excel = pasta_arquivo / f"{arquivo.stem}.xlsx"
            dest_pdf = pasta_arquivo / arquivo.name

            shutil.copy2(lancamento, dest_lancamento)

            planilha_lancamento(df, dest_lancamento)

            df = totalizador(df)

            df.to_excel(dest_excel, index=False)

            shutil.move(str(arquivo), str(dest_pdf))

            print(f"PDF convertido com sucesso: {arquivo.name}")
            print(f"Arquivos salvos em: {pasta_arquivo}")



In [ ]:
from pathlib import Path
from typing import Optional
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

class GoogleDrive():
    def __init__(self, GOOGLE_APPLICATION_CREDENTIALS):
        self.SCOPES = ["https://www.googleapis.com/auth/drive"]
        self.GOOGLE_APPLICATION_CREDENTIALS = GOOGLE_APPLICATION_CREDENTIALS

    def _service(self):
        credentials = Credentials.from_service_account_file(
            self.GOOGLE_APPLICATION_CREDENTIALS,
            scopes=self.SCOPES
        )
        return build("drive", "v3", credentials=credentials)
    
    def list_folder(self, folder_id, folder_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType = '{folder_type}' "
            f"and trashed = false"
        )
        pastas = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageToken = page_token
            ).execute()
            pastas.extend(response.get("files", []))
            page_token = response.get("nextPageToken")

            if not page_token:
                break

        return pastas
    
    def list_folder_by_name(self, folder_id, name_folder, folder_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and name = '{name_folder}' "
            f"and mimeType = '{folder_type}' "
            f"and trashed = false"
        )
        response = service.files().list(
            q=query,
            spaces="drive",
            fields="files(id, name, mimeType)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        pastas = response.get("files", [])
        return pastas[0] if pastas else None

    def list_files(self, folder_id):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType != 'application/vnd.google-apps.folder' "
            f"and trashed = false"
        )

        arquivos = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType, size, modifiedTime)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageToken=page_token
            ).execute()
            arquivos.extend(response.get("files", []))
            page_token = response.get("nextPageToken")
            if not page_token:
                break
        return arquivos
    
    def search_file_by_name(self, folder_id, name_file):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and name = '{name_file}' "
            f"and trashed = false"
        )
        response = service.files().list(
            q=query,
            spaces="drive",
            fields="files(id, name, mimeType, size, modifiedTime)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        arquivos = response.get("files", [])
        return arquivos[0] if arquivos else None
    
    def pdfs(self, folder_id, pdf_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType = '{pdf_type}' "
            f"and trashed = false"
        )
        arquivos = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType, size, modifiedTime)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            arquivos.extend(response.get("files", []))
            page_token = response.get("nextPageToken")
            if not page_token:
                break
        return arquivos
    
    def create_folder(self, name_folder, type_folder, folder_id_pai=None):
        service = self._service()
        metadata = {
            "name": name_folder,
            "mimeType": type_folder
        }

        if folder_id_pai:
            metadata["parents"] = [folder_id_pai]

        pasta = service.files().create(
            body=metadata,
            fields="id, name, mimeType",
            supportsAllDrives=True
        ).execute()
        return pasta
    
    def get_or_create_folder(self, folder_id_pai, name_folder):
        folder_type = "application/vnd.google-apps.folder"

        pasta = self.list_folder_by_name(
            folder_id=folder_id_pai,
            name_folder=name_folder,
            folder_type=folder_type
        )

        if pasta:
            return pasta

        return self.create_folder(
            name_folder=name_folder,
            type_folder=folder_type,
            folder_id_pai=folder_id_pai
        )
    
    def move_file(self, file_id, folder_id_destino):
        service = self._service()

        arquivo = service.files().get(
            fileId=file_id,
            fields="parents",
            supportsAllDrives=True
        ).execute()

        parents_atuais = ",".join(arquivo.get("parents", []))

        arquivo_movido = service.files().update(
            fileId=file_id,
            addParents=folder_id_destino,
            removeParents=parents_atuais,
            fields="id, name, parents",
            supportsAllDrives=True
        ).execute()

        return arquivo_movido
    
    def download(self, file_id, destino_local):
        service = self._service()
        destino = Path(destino_local)
        destino.parent.mkdir(parents=True, exist_ok=True)

        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(destino, "wb") as arquivo_local:
            downloader = MediaIoBaseDownload(arquivo_local, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        return str(destino)
    
    def upload(self, caminho_local, folder_id_destino, type_file, name_drive: Optional[str] = None):
        service = self._service()
        caminho = Path(caminho_local)
        if not caminho.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho_local}")
        
        nome_final = name_drive or caminho.name
        metadata = {
            "name": nome_final,
            "parents": [folder_id_destino]
        }
        media = MediaFileUpload(
            filename=str(caminho),
            mimetype=type_file,
            resumable=True
        )

        arquivo = service.files().create(
            body=metadata,
            media_body=media,
            fields="id, name, mimeType, size, modifiedTime",
            supportsAllDrives=True
        ).execute()
        return arquivo
    
